In [4]:
import os
import base64
from dotenv import load_dotenv
from google.cloud import aiplatform
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "key.json"
# 1. Load configuration
load_dotenv()

PROJECT_ID = os.getenv("PROJECT_ID")
REGION = os.getenv("REGION")
ENDPOINT_ID = os.getenv("ENDPOINT_ID")

# 2. Initialize Vertex AI
aiplatform.init(project=PROJECT_ID, location=REGION)
endpoint_name = f"projects/{PROJECT_ID}/locations/{REGION}/endpoints/{ENDPOINT_ID}"
endpoint = aiplatform.Endpoint(endpoint_name)

def encode_image(image_path):
    """Encodes a local image to base64 string with the proper data prefix."""
    with open(image_path, "rb") as image_file:
        b64_string = base64.b64encode(image_file.read()).decode("utf-8")
        # Standard OpenAI format for base64 images
        return f"data:image/jpeg;base64,{b64_string}"

def predict_text_only(prompt: str, system_prompt: str = "You are an expert medical doctor.", max_tokens: int = 1024):
    """Function 1: Text-only prediction."""
    instances = [{
        "@requestFormat": "chatCompletions",
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {"role": "user", "content": [{"type": "text", "text": prompt}]}
        ],
        "max_tokens": max_tokens
    }]
    try:
        response = endpoint.predict(instances=instances)
        return response.predictions if response.predictions else None
    except Exception as e:
        print(f"Error: {e}")
        return None

def predict_multimodal(prompt: str, image_paths: list, system_prompt: str = "You are an expert medical doctor.", max_tokens: int = 2048):
    """Function 2: Text + Multiple Images prediction."""
    
    # Create the user content list starting with the text prompt
    user_content = [{"type": "text", "text": prompt}]
    
    # Append each image to the user content
    for path in image_paths:
        user_content.append({
            "type": "image_url",
            "image_url": {"url": encode_image(path)}
        })

    instances = [{
        "@requestFormat": "chatCompletions",
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {"role": "user", "content": user_content}
        ],
        "max_tokens": max_tokens
    }]

    try:
        response = endpoint.predict(instances=instances)
        return response.predictions if response.predictions else None
    except Exception as e:
        print(f"Error: {e}")
        return None

# --- Example Usage ---
# images = ["lab_report_p1.jpg", "lab_report_p2.jpg"]
# result = predict_multimodal("Summarize these reports.", images)
# print(result)


print(predict_text_only("What is the capital of France?"))

{'choices': [{'index': 0, 'message': {'content': '<unused94>thought\n1.  **Identify the core question:** The user is asking for the capital of France.\n2.  **Access knowledge about France:** Recall or look up information about France.\n3.  **Extract the capital city:** The capital city of France is Paris.\n4.  **Formulate the answer:** State the capital city clearly and concisely.<unused95>The capital of France is **Paris**.', 'role': 'assistant'}}], 'created': 1769250316, 'id': 'd1d425d7-5eb7-4ebc-8013-8dfa11563885', 'model': 'placeholder', 'object': 'chat.completion', 'usage': {'completion_tokens': 86, 'prompt_tokens': 25, 'total_tokens': 111}}
